# RL Practical Exam: One Template, Twelve Labs

Every lab uses the **same four blocks in the same order**. Only three things ever change: the `POLICY` block, the `ENVIRONMENT` block and the `EQUATION` block.
The names are identical everywhere: `state, action, next_state, reward, alpha, gamma, epsilon, Q, V`.
Run the cells top to bottom (Google Colab ready).

**Contents**

| Lab | Topic | Type | Library |
|-----|-------|------|---------|
| 1 | Epsilon-greedy for choosing an advertisement | Bandit (no states) | NumPy |
| 2 | State value function estimation, TD(0) | Prediction, bootstrapping | NumPy |
| 3 | Policy Iteration | Planning, model known | NumPy |
| 4 | Monte Carlo prediction | Prediction, full episodes | NumPy |
| 5 | Q-Learning | Control, off-policy | NumPy |
| 6 | SARSA | Control, on-policy | NumPy |
| 7 | Deep Q-Network on CartPole | Control, value-based, network | PyTorch + Gymnasium |
| 8 | TD-Gammon style learning, TD(lambda) + network | Prediction + control by self-play | PyTorch |
| 9 | REINFORCE on CartPole | Policy gradient | PyTorch + Gymnasium |
| 10 | Actor-Critic (one-step, every step) | Policy gradient + critic | PyTorch + Gymnasium |
| 11 | Advantage Actor-Critic, A2C (per episode) | Policy gradient + critic | PyTorch + Gymnasium |
| 12 | Proximal Policy Optimization (PPO) | Policy gradient, clipped | PyTorch + Gymnasium |

**How to use this in the exam (5-step routine)**
1. Write the four block comments first (`# 1. SETUP` ... `# 4. OUTPUT`). That is already 30% of the program.
2. Fill SETUP (`alpha, gamma, epsilon, episodes`) and the STATES AND ACTIONS table or network.
3. Write the loop, then fill the three slots in this order: POLICY, ENVIRONMENT, EQUATION.
4. Print something meaningful in OUTPUT (the table, the path, the average reward). Examiners look at the output first.
5. If the question says "compare" or "show", add a second table or a print of the first 100 vs last 100 episodes.

> Note: your list has two actor-critic labs (10 and 11). I made **Lab 10 = one-step TD actor-critic (update every step)** and **Lab 11 = A2C (update after each episode using returns, plus an entropy bonus)**. If your lab manual splits them differently, the code still works because the template is the same. Only the EQUATION block moves.

## The Template (memorize this once)

```python
# 1. SETUP                      -> alpha, gamma, epsilon, episodes
# 2. STATES AND ACTIONS         -> states, goal, moves (actions), Q or V table (or networks)
# 3. TRAINING LOOP
for episode in range(episodes):
    state = start
    while state != goal:              # (or: for step in range(...), or: while not done)
        # POLICY                      -> how to choose the action
        # ENVIRONMENT                 -> next_state, reward
        # EQUATION                    -> the learning update   <-- changes per algorithm
        state = next_state
# 4. OUTPUT                     -> print the table / path / reward
```

**What goes in each slot**

| Slot | Tabular (Labs 1-6) | Neural (Labs 7-12) |
|------|--------------------|--------------------|
| SETUP | `alpha, gamma, epsilon, episodes` | `gamma, lr, episodes` (+ `epsilon` for DQN) |
| STATES AND ACTIONS | `Q = np.zeros(...)` or `V = np.zeros(...)` | `nn.Sequential(...)` networks + `optimizer` |
| POLICY | random / epsilon-greedy / `argmax` | `Categorical(actor(x)).sample()` or epsilon-greedy on `model(x)` |
| ENVIRONMENT | `next_state, reward` from a few lines or a `step()` function | `env.step(action)` from Gymnasium |
| EQUATION | one update line | a `loss`, then `zero_grad() / backward() / step()` |

**The equations (the only thing that really changes)**
```python
# Bandit        Q[action] += (reward - Q[action]) / N[action]                      no next state
# TD(0)         V[state]  += alpha * (reward + gamma * V[next_state] - V[state])
# Monte Carlo   V[state]  += (G - V[state]) / N[state]                              G = return of the whole episode
# Policy Iter.  V[s] = r + gamma * V[s2]  (evaluate), then policy[s] = argmax_a (r + gamma * V[s2])  (improve)
# Q-Learning    Q[s, a]   += alpha * (r + gamma * max(Q[s2]) - Q[s, a])
# SARSA         Q[s, a]   += alpha * (r + gamma * Q[s2, a2] - Q[s, a])
# DQN           loss = (Q(s, a) - (r + gamma * max Q_target(s2))) ** 2
# TD(lambda)    trace = gamma*lambda*trace + grad V(s);   w += alpha * (V(s2) - V(s)) * trace
# REINFORCE     loss = -(log_prob * G)
# Actor-Critic  adv = r + gamma * V(s2) - V(s);   loss = -log_prob * adv + adv ** 2
# A2C           adv = G - V(s);   loss = -log_prob * adv + 0.5 * adv ** 2 - 0.01 * entropy
# PPO           ratio = exp(new_log_prob - old_log_prob);   loss = -min(ratio * adv, clip(ratio, 0.8, 1.2) * adv)
```

**Symbols**: `alpha` = learning rate, `gamma` = discount (how much the future matters), `epsilon` = exploration probability, `G` = discounted return = `r1 + gamma*r2 + gamma^2*r3 + ...`.

## Cheat Sheet: Snippets You Must Be Able to Write From Memory

```python
# Epsilon-greedy (tabular)
if random.random() < epsilon:
    action = random.randint(0, n_actions - 1)         # explore
else:
    action = np.argmax(Q[state])                       # exploit

# Discounted returns (used in 4, 9, 11, 12)
G = 0
returns = []
for reward in reversed(rewards):
    G = reward + gamma * G
    returns.insert(0, G)

# Policy network + sampling (used in 9, 10, 11, 12)
actor = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 2), nn.Softmax(dim=-1))
dist = torch.distributions.Categorical(actor(x))
action = dist.sample()                                 # a tensor
log_prob = dist.log_prob(action)                       # used in the loss
env.step(action.item())                                # .item() converts tensor -> int

# The 3-line PyTorch update (always the same)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# Gymnasium CartPole (note: reset() returns 2 things, step() returns 5 things)
env = gym.make("CartPole-v1")
state, _ = env.reset()
next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated
```

**Why these details matter**
- `.detach()` on a target or advantage stops gradients flowing into the wrong network. Forgetting it is the most common bug.
- `terminated` means the pole really fell (no future reward, so the target has no `gamma * V(next)` term). `truncated` means the 500-step time limit hit (the future was not zero, so do not treat it as terminal).
- Dividing states by a constant (`state / 100`) keeps network inputs small and training stable.

## Lab 1. Epsilon-Greedy for Choosing an Advertisement (Multi-Armed Bandit)

**Theory**
- Each ad is an **arm**. Showing an ad is an **action**. A click gives `reward = 1`, no click gives `0`.
- The true click rate of every ad is hidden. We estimate it as `Q[ad]` from experience.
- **Exploration vs exploitation**: always showing the best-looking ad (greedy) can lock onto a wrong ad. Showing random ads forever wastes clicks. Epsilon-greedy explores with probability `epsilon` and exploits (`argmax Q`) with probability `1 - epsilon`.
- There is **no state** and no next state, so there is no `gamma` and no `Q[next_state]`.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy over ads |
| ENVIRONMENT | reward = 1 with probability `true_ctr[action]`, else 0 |
| EQUATION | `Q[action] += (reward - Q[action]) / N[action]` (sample average) |

**What to keep / what changes (this is the base lab, the rest build on it)**
- Keep: the epsilon-greedy lines. They are reused in Labs 5, 6, 7.
- Change later: for Lab 5 we add a state and the `gamma * max Q[next_state]` term.
- Why `1 / N[action]`: it makes `Q` the exact average of all rewards seen. A constant `alpha` also works (it weights recent rewards more, which is better when click rates drift).

**Common mistakes**: `Q` as a 2D table (it is 1D, one number per ad); forgetting to update `N`; picking `argmax` on ties (fine, but exploration fixes it).

**Expected output**: `Q` close to `true_ctr`; ad index 3 (CTR 0.20) is shown most often; total clicks is higher than random play (~0.11 * 2000 = 220).

In [ ]:
import numpy as np
import random

# 1. SETUP
epsilon = 0.1
episodes = 2000                      # each episode = show one ad to one user

# 2. STATES AND ACTIONS
true_ctr = [0.05, 0.12, 0.08, 0.20, 0.10]   # hidden click probability of each ad
n_actions = len(true_ctr)                   # action = which ad to show
Q = np.zeros(n_actions)                     # estimated click rate of each ad
N = np.zeros(n_actions)                     # how many times each ad was shown
total_clicks = 0

# 3. TRAINING LOOP
for episode in range(episodes):
    # POLICY: epsilon-greedy
    if random.random() < epsilon:
        action = random.randint(0, n_actions - 1)
    else:
        action = np.argmax(Q)

    # ENVIRONMENT: the user clicks (1) or not (0)
    reward = 1 if random.random() < true_ctr[action] else 0
    N[action] += 1
    total_clicks += reward

    # EQUATION: bandit (sample average, no next state)
    Q[action] += (reward - Q[action]) / N[action]

# 4. OUTPUT
print("Estimated CTR:", np.round(Q, 3))
print("True CTR:     ", true_ctr)
print("Times shown:  ", N)
print("Best ad:", np.argmax(Q), "| Total clicks:", total_clicks)

## Lab 2. State Value Function Estimation (TD(0) on a Random Walk)

**Theory**
- The **state value** `V(s)` = expected total future reward when starting in `s` and following a fixed policy. This is **prediction** (evaluate a policy), not control (find a policy).
- **TD(0)** learns `V` from single steps. It moves `V[state]` toward the **TD target** `reward + gamma * V[next_state]`. The gap between them is the **TD error**.
- **Bootstrapping** = using the current estimate `V[next_state]` inside the update (learning a guess from a guess).
- World: states `0..6`, in a line. `0` and `6` are terminal. The agent starts at `3` and moves left or right with equal probability. Reaching `6` gives reward `1`, everything else `0`. With `gamma = 1` the true values are `1/6, 2/6, ..., 5/6` for states `1..5`, so you can check your answer.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | random left/right (the policy being evaluated) |
| ENVIRONMENT | `next_state = state + action`, reward 1 only on reaching state 6 |
| EQUATION | `V[state] += alpha * (reward + gamma * V[next_state] - V[state])` |

**Keep / change**
- Keep the template. There is a `V` table (one number per state) instead of `Q` (no action index).
- Terminal values stay `0` because they are never updated, so `V[next_state]` is `0` at the end. This is what makes the reward propagate backward.

**Expected output**: `V[1:6]` close to `[0.17, 0.33, 0.5, 0.67, 0.83]`.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.05
gamma = 1.0
episodes = 2000

# 2. STATES AND ACTIONS
n_states = 7
start = 3
terminals = [0, 6]
moves = [-1, 1]                                         # left, right
V = np.zeros(n_states)

# 3. TRAINING LOOP
for episode in range(episodes):
    state = start
    while state not in terminals:
        # POLICY: random
        action = random.choice(moves)

        # ENVIRONMENT
        next_state = state + action
        reward = 1 if next_state == 6 else 0

        # EQUATION: TD(0)
        V[state] += alpha * (
            reward + gamma * V[next_state] - V[state]
        )

        state = next_state

# 4. OUTPUT
print("Estimated V:", np.round(V[1:6], 2))
print("True V:     ", np.round([1/6, 2/6, 3/6, 4/6, 5/6], 2))

## Lab 3. Policy Iteration (Dynamic Programming)

**Theory**
- Policy iteration needs a **known model** (we know exactly where each action leads), so there is no sampling and no `alpha` or `epsilon`.
- It repeats two steps until the policy stops changing:
  1. **Policy evaluation**: compute `V` for the current policy by sweeping `V[s] = r + gamma * V[s2]` until the change is tiny (`delta < theta`).
  2. **Policy improvement**: for each state pick `argmax over actions of (r + gamma * V[s2])`.
- It is guaranteed to reach the optimal policy for a finite problem.
- World: 4x4 grid, states `0..15`. States `0` and `15` are terminal. Every move costs `-1`. Moving off the grid leaves you in place.

**Slots (the template changes shape here)**
| Slot | This lab |
|------|----------|
| POLICY | a table `policy[state]` that we improve (deterministic) |
| ENVIRONMENT | the `step(state, action)` function (the model) |
| EQUATION | evaluation sweep, then greedy improvement |

**Keep / change vs the learning labs**
- Change: no episodes of experience. The loop is `while True:` evaluate, improve, stop when stable.
- Change: we sweep **all** states each time instead of following one trajectory.
- Why `gamma = 0.9`: the starting policy (always up) never reaches a terminal from some states, so with `gamma = 1` evaluation would diverge. A discount keeps it finite.
- Why `policy_stable` uses `+ 1e-9`: ties between equally good actions would make the policy flip forever.

**Expected output**: arrows that point toward the nearest corner terminal (0 or 15).

In [ ]:
import numpy as np

# 1. SETUP
gamma = 0.9
theta = 1e-6                                            # stop evaluation when the change is this small

# 2. STATES AND ACTIONS
n = 4
n_states = n * n
terminals = [0, 15]
moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]              # up, down, left, right
arrows = ["^", "v", "<", ">"]
V = np.zeros(n_states)
policy = np.zeros(n_states, dtype=int)                  # start: always up

# ENVIRONMENT: the model (we know where every action leads)
def step(state, action):
    row, col = divmod(state, n)
    row = max(0, min(n - 1, row + moves[action][0]))
    col = max(0, min(n - 1, col + moves[action][1]))
    return row * n + col, -1                            # next_state, reward

# 3. TRAINING LOOP
while True:
    # EQUATION part 1: policy evaluation
    while True:
        delta = 0
        for state in range(n_states):
            if state in terminals:
                continue
            next_state, reward = step(state, policy[state])
            new_v = reward + gamma * V[next_state]
            delta = max(delta, abs(new_v - V[state]))
            V[state] = new_v
        if delta < theta:
            break

    # EQUATION part 2: policy improvement
    policy_stable = True
    for state in range(n_states):
        if state in terminals:
            continue
        q_values = []
        for action in range(4):
            next_state, reward = step(state, action)
            q_values.append(reward + gamma * V[next_state])
        best = int(np.argmax(q_values))
        if q_values[best] > q_values[policy[state]] + 1e-9:   # change only if strictly better
            policy[state] = best
            policy_stable = False

    if policy_stable:
        break

# 4. OUTPUT
print("Value function:\n", np.round(V.reshape(n, n), 2))
print("Optimal policy:")
for row in range(n):
    print(" ".join("T" if row * n + col in terminals else arrows[policy[row * n + col]] for col in range(n)))

## Lab 4. Monte Carlo Prediction

**Theory**
- Monte Carlo (MC) learns `V` from **complete episodes**. It waits until the episode ends, then computes the real return `G` for each visited state and averages it.
- **No bootstrapping**: it never uses `V[next_state]`. The target is the true observed return. This makes it **unbiased but high variance**, and it only works for episodic tasks.
- **First-visit MC** updates a state only the first time it appears in an episode. **Every-visit MC** updates at every appearance. Both converge to the true `V`.
- Same random-walk world as Lab 2, so you can compare with TD(0).

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | random left/right |
| ENVIRONMENT | same random walk, but we **store** `(state, reward)` for the whole episode |
| EQUATION | **after** the episode: go backward computing `G`, then `V[s] += (G - V[s]) / N[s]` |

**Keep / change vs Lab 2 (TD(0))**
- Keep: policy and environment.
- Change: the equation moves out of the `while` loop to after the episode (like REINFORCE in Lab 9).
- Change: target is `G` (real return) instead of `reward + gamma * V[next_state]`.
- Why backward: `G = reward + gamma * G` builds the return from the last step to the first in one pass.

**TD vs MC (viva favourite)**: TD updates every step, has low variance but some bias, works for continuing tasks. MC updates once per episode, has no bias but high variance, needs terminal states.

**Expected output**: `V[1:6]` close to `[0.17, 0.33, 0.5, 0.67, 0.83]`.

In [ ]:
import numpy as np
import random

# 1. SETUP
gamma = 1.0
episodes = 3000

# 2. STATES AND ACTIONS
n_states = 7
start = 3
terminals = [0, 6]
moves = [-1, 1]                                         # left, right
V = np.zeros(n_states)
N = np.zeros(n_states)                                  # visit counts

# 3. TRAINING LOOP
for episode in range(episodes):
    state = start
    history = []                                        # list of (state, reward)

    while state not in terminals:
        # POLICY: random
        action = random.choice(moves)

        # ENVIRONMENT
        next_state = state + action
        reward = 1 if next_state == 6 else 0
        history.append((state, reward))

        state = next_state

    # EQUATION: first-visit Monte Carlo (after the episode)
    G = 0
    for t in reversed(range(len(history))):
        s, r = history[t]
        G = r + gamma * G
        if s not in [x[0] for x in history[:t]]:        # first visit only
            N[s] += 1
            V[s] += (G - V[s]) / N[s]

# 4. OUTPUT
print("Estimated V:", np.round(V[1:6], 2))
print("True V:     ", np.round([1/6, 2/6, 3/6, 4/6, 5/6], 2))

## Lab 5. Q-Learning (Grid World)

**Theory**
- Now we want to **find a good policy** (control), so we learn the action value `Q[state, action]` instead of `V`.
- **Q-Learning is off-policy**: it behaves epsilon-greedily (explores) but learns about the **greedy** policy, because the target uses `max` over the next actions.
- It converges to the optimal `Q*` as long as every state-action pair keeps being visited.
- World: 4x4 grid, start `0`, goal `15`. Each step costs `-1`, reaching the goal gives `+10`. Actions: up, down, left, right.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy on `Q[state]` |
| ENVIRONMENT | `step(state, action)` |
| EQUATION | `Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])` |

**Keep / change vs earlier labs**
- Keep: epsilon-greedy lines from Lab 1.
- Change vs Lab 2 (TD(0)): the table gets an action axis (`Q[state, action]`) and the target takes a `max` over next actions.
- Change vs Lab 1: there is now a state, so `gamma * max Q[next_state]` appears.

**Common mistakes**: Q-table shape must be `(n_states, n_actions)` (here 16 by 4); stopping the greedy test run with no step limit.

**Expected output**: a greedy path of 6 moves from state 0 to 15, e.g. `[0, 1, 2, 3, 7, 11, 15]` or any equal-length route.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
epsilon = 0.1
episodes = 500

# 2. STATES AND ACTIONS
n = 4
n_states = n * n
goal = 15
moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]              # up, down, left, right
n_actions = len(moves)
Q = np.zeros((n_states, n_actions))

def step(state, action):                                # ENVIRONMENT
    row, col = divmod(state, n)
    row = max(0, min(n - 1, row + moves[action][0]))
    col = max(0, min(n - 1, col + moves[action][1]))
    next_state = row * n + col
    reward = 10 if next_state == goal else -1
    return next_state, reward

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0
    while state != goal:
        # POLICY: epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, n_actions - 1)
        else:
            action = np.argmax(Q[state])

        # ENVIRONMENT
        next_state, reward = step(state, action)

        # EQUATION: Q-Learning
        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state])
            - Q[state, action]
        )

        state = next_state

# 4. OUTPUT
print("Max Q per state:\n", np.round(Q.max(axis=1).reshape(n, n), 1))

state = 0
path = [state]
for _ in range(20):                                     # greedy test run
    state, _r = step(state, np.argmax(Q[state]))
    path.append(state)
    if state == goal:
        break
print("Path:", path)

## Lab 6. SARSA

**Theory**
- SARSA = **S**tate, **A**ction, **R**eward, next **S**tate, next **A**ction (the five things in one update).
- **On-policy**: the target uses the action the agent will **actually take** next (`next_action`, chosen epsilon-greedily), not the best one. So it learns the value of the policy it is following, exploration included.
- Q-Learning learns the optimal greedy policy. SARSA learns a safer policy that accounts for its own exploration (the classic cliff-walking example: SARSA stays away from the cliff edge, Q-Learning walks along it).
- Same grid world as Lab 5.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy, used **twice**: first action before the loop, then `next_action` inside |
| ENVIRONMENT | same `step()` as Lab 5 |
| EQUATION | `Q[s, a] += alpha * (r + gamma * Q[s2, a2] - Q[s, a])` |

**Exactly three changes from Lab 5 (marked `# SARSA` in the code)**
1. Choose the first `action` before the `while` loop.
2. Choose `next_action` right after the environment step, before the update.
3. In the equation use `Q[next_state, next_action]` instead of `np.max(Q[next_state])`, then set `action = next_action` at the end.

**Why choose `next_action` before the update**: the update needs it. This is the single most-asked difference.

**Expected output**: the same 6-move path as Lab 5.

In [ ]:
import numpy as np
import random

# 1. SETUP
alpha = 0.1
gamma = 0.9
epsilon = 0.1
episodes = 500

# 2. STATES AND ACTIONS
n = 4
n_states = n * n
goal = 15
moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]              # up, down, left, right
n_actions = len(moves)
Q = np.zeros((n_states, n_actions))

def step(state, action):                                # ENVIRONMENT
    row, col = divmod(state, n)
    row = max(0, min(n - 1, row + moves[action][0]))
    col = max(0, min(n - 1, col + moves[action][1]))
    next_state = row * n + col
    reward = 10 if next_state == goal else -1
    return next_state, reward

def choose(state):                                      # POLICY: epsilon-greedy
    if random.random() < epsilon:
        return random.randint(0, n_actions - 1)
    return np.argmax(Q[state])

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0
    action = choose(state)                              # SARSA 1: first action before the loop

    while state != goal:
        # ENVIRONMENT
        next_state, reward = step(state, action)

        # POLICY: next action
        next_action = choose(next_state)                # SARSA 2: choose next_action now

        # EQUATION: SARSA
        Q[state, action] += alpha * (
            reward + gamma * Q[next_state, next_action] # SARSA 3: Q[next_state, next_action], no max
            - Q[state, action]
        )

        state = next_state
        action = next_action                            # carry the action forward

# 4. OUTPUT
print("Max Q per state:\n", np.round(Q.max(axis=1).reshape(n, n), 1))

state = 0
path = [state]
for _ in range(20):                                     # greedy test run
    state, _r = step(state, np.argmax(Q[state]))
    path.append(state)
    if state == goal:
        break
print("Path:", path)

## Lab 7. Deep Q-Network (DQN) on CartPole

**Theory**
- CartPole: balance a pole on a cart. State = 4 real numbers (cart position, cart velocity, pole angle, pole angular velocity). 2 actions (push left, push right). Reward `+1` per step the pole stays up (max 500).
- A table is impossible for continuous states, so a **neural network** maps `state -> [Q(left), Q(right)]`.
- DQN = Q-Learning (Lab 5) + a network + two stabilizers:
  1. **Replay buffer**: store `(state, action, reward, next_state, terminated)` and train on a random mini-batch. This breaks the correlation between consecutive steps.
  2. **Target network**: a slowly updated copy of the network used to compute the target, so the target does not move every step.
- Epsilon starts at `1.0` and decays to `0.05` (explore a lot first, then exploit).

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy: `model(x).argmax()` |
| ENVIRONMENT | `env.step(action)` |
| EQUATION | `loss = (Q(s,a) - (r + gamma * max Q_target(s2) * (1 - terminated))) ** 2` |

**Keep / change vs Lab 5 (Q-Learning)**
- Keep: epsilon-greedy policy and the target `r + gamma * max Q(next)`.
- Change: table becomes `model`, the update line becomes a `loss` plus `zero_grad / backward / step`.
- Add: `buffer`, `target_model`, and `(1 - terminated)` so a fallen pole has no future value.
- Why `gather(1, actions)`: picks the Q-value of the action that was actually taken from the network's 2 outputs.
- Why `torch.no_grad()` on the target: the target is a label, not something to train.

**Expected output**: average reward rises from about 20 to 150+ (it is noisy, DQN often spikes then dips).

In [ ]:
!pip install -q gymnasium

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import random
from collections import deque

# 1. SETUP
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.99
episodes = 300
batch_size = 64
lr = 0.001

# 2. STATES AND ACTIONS
env = gym.make("CartPole-v1")                           # state: 4 numbers, 2 actions
n_actions = 2
model = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, n_actions))
target_model = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, n_actions))
target_model.load_state_dict(model.state_dict())        # start as an exact copy
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
buffer = deque(maxlen=10000)                            # replay buffer
totals = []

# 3. TRAINING LOOP
for episode in range(episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        # POLICY: epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, n_actions - 1)
        else:
            action = model(torch.tensor(state, dtype=torch.float32)).argmax().item()

        # ENVIRONMENT
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        buffer.append((state, action, reward, next_state, terminated))
        state = next_state
        total_reward += reward

        # EQUATION: DQN (learn from a random mini-batch)
        if len(buffer) >= batch_size:
            batch = random.sample(buffer, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            states = torch.tensor(np.array(states), dtype=torch.float32)
            actions = torch.tensor(actions).unsqueeze(1)
            rewards = torch.tensor(rewards, dtype=torch.float32)
            next_states = torch.tensor(np.array(next_states), dtype=torch.float32)
            dones = torch.tensor(dones, dtype=torch.float32)

            q_value = model(states).gather(1, actions).squeeze(1)
            with torch.no_grad():
                target = rewards + gamma * target_model(next_states).max(1)[0] * (1 - dones)
            loss = nn.functional.mse_loss(q_value, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)  # explore less over time
    if (episode + 1) % 10 == 0:
        target_model.load_state_dict(model.state_dict()) # refresh the target network

    totals.append(total_reward)
    if (episode + 1) % 20 == 0:
        print("Episode", episode + 1, "Avg reward (last 20):", sum(totals[-20:]) / 20)

# 4. OUTPUT
print("Avg reward, first 20 episodes:", sum(totals[:20]) / 20)
print("Avg reward, last 20 episodes: ", sum(totals[-20:]) / 20)

## Lab 8. TD-Gammon Style Learning (TD(lambda) + Neural Network + Self-Play)

**Theory**
- TD-Gammon (Tesauro, 1992) learned to play backgammon at expert level using only three ideas:
  1. A **neural network** that estimates `V(board)` = probability that player X wins.
  2. **TD(lambda)** learning: after each move, nudge the network so `V(current board)` moves toward `V(next board)`. The only real reward is at the end (win = 1, loss = 0).
  3. **Self-play**: the same network plays both sides. X picks the move with the **highest** value, O picks the **lowest**.
- **Eligibility trace** `e`: a running memory of which weights recently influenced the value. `e = gamma * lambda * e + gradient of V(state)`. The update `w += alpha * delta * e` pushes credit back to earlier moves. `lambda = 0` is plain TD(0), `lambda = 1` is Monte Carlo.
- Backgammon is too big for an exam, so we use **Tic-Tac-Toe** with the identical algorithm. Board is encoded as 9 numbers (`X = +1`, `O = -1`, empty `= 0`).

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | look one move ahead: evaluate the board after each legal move, X takes max, O takes min (plus a little epsilon) |
| ENVIRONMENT | place the mark, check for a win or full board, reward only at the end |
| EQUATION | `delta = V(new) - V(prev)`; `w += alpha * delta * trace`; `trace = gamma*lambda*trace + grad V(new)` |

**Keep / change vs Lab 2 (TD(0))**
- Keep: the TD error `target - V(previous)`. It is the same idea.
- Change: `V` is a network, so "update `V[state]`" becomes "update the weights using the gradient".
- Add: `traces` (one tensor per weight) and **self-play** with two players sharing one network.
- Why compare with the previous value: we do not know the true outcome until the game ends, so each position learns from the next one.

**Expected output**: the trained X (always playing the greedy move) beats a random O in roughly 85 to 95 percent of games.

In [ ]:
import torch
import torch.nn as nn
import random

# 1. SETUP
alpha = 0.1
gamma = 1.0
lam = 0.7                                               # lambda: trace decay
epsilon = 0.1
episodes = 3000

# 2. STATES AND ACTIONS
lines = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
net = nn.Sequential(nn.Linear(9, 16), nn.Sigmoid(), nn.Linear(16, 1), nn.Sigmoid())   # V(board) = P(X wins)

def encode(board):                                      # board string -> 9 numbers
    return torch.tensor([1.0 if c == "X" else -1.0 if c == "O" else 0.0 for c in board])

def play(board, square, player):                        # return the new board
    return board[:square] + player + board[square + 1:]

def winner(board):
    for a, b, c in lines:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]
    return None

def best_move(board, player):                           # one-move lookahead on the network
    moves = [i for i in range(9) if board[i] == " "]
    values = [net(encode(play(board, m, player))).item() for m in moves]
    return moves[values.index(max(values))] if player == "X" else moves[values.index(min(values))]

# 3. TRAINING LOOP (self-play)
for episode in range(episodes):
    board = " " * 9
    player = "X"
    traces = [torch.zeros_like(p) for p in net.parameters()]
    prev_value = None
    done = False

    while not done:
        # POLICY: epsilon-greedy using the network
        if random.random() < epsilon:
            action = random.choice([i for i in range(9) if board[i] == " "])
        else:
            action = best_move(board, player)

        # ENVIRONMENT
        board = play(board, action, player)
        win = winner(board)
        done = win is not None or " " not in board

        # EQUATION: TD(lambda)
        value = net(encode(board))
        net.zero_grad()
        value.backward()                                # gradient of V(new board)

        if done:
            target = 1.0 if win == "X" else 0.0 if win == "O" else 0.5   # real outcome
        else:
            target = value.item()                       # bootstrap from the next board

        if prev_value is not None:
            delta = gamma * target - prev_value         # TD error
            with torch.no_grad():
                for p, e in zip(net.parameters(), traces):
                    p += alpha * delta * e              # w += alpha * delta * trace

        for p, e in zip(net.parameters(), traces):
            e.mul_(gamma * lam).add_(p.grad)            # trace = gamma*lambda*trace + grad

        prev_value = value.item()
        player = "O" if player == "X" else "X"

# 4. OUTPUT: trained X (greedy) vs a random O
wins = draws = losses = 0
for game in range(500):
    board = " " * 9
    player = "X"
    done = False
    while not done:
        if player == "X":
            action = best_move(board, "X")
        else:
            action = random.choice([i for i in range(9) if board[i] == " "])
        board = play(board, action, player)
        win = winner(board)
        done = win is not None or " " not in board
        player = "O" if player == "X" else "X"
    if win == "X":
        wins += 1
    elif win == "O":
        losses += 1
    else:
        draws += 1
print("Trained X vs random O -> wins:", wins, "draws:", draws, "losses:", losses)
print("V(empty-ish board, X in centre):", round(net(encode(" " * 4 + "X" + " " * 4)).item(), 3))

## Lab 9. CartPole Using REINFORCE (Monte Carlo Policy Gradient)

**Theory**
- Value-based methods learn `Q` then act greedily. **Policy-gradient** methods learn the **policy itself**: a network that outputs action probabilities `pi(action | state)`.
- **REINFORCE** idea: play a whole episode, then make actions that led to a high return **more likely** and actions that led to a low return **less likely**.
- Gradient: `loss = -log_prob(action) * G`. Minimizing this raises `log_prob` when `G` is high.
- It is Monte Carlo (Lab 4) applied to a policy: it uses the full return `G` and updates after the episode.
- **Return normalization** (subtract the mean, divide by std) acts as a simple baseline and reduces variance. This is why the code has the `(returns - mean) / std` line.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | `Categorical(actor(x)).sample()`, **save** `log_prob` |
| ENVIRONMENT | `env.step(action.item())`, **save** `reward` |
| EQUATION | after the episode: compute `G`, `loss = -(log_probs * returns).mean()` |

**Keep / change vs Lab 7 (DQN) and Lab 4 (MC)**
- Keep from Lab 4: backward return computation after the episode.
- Change from Lab 7: no epsilon, no replay buffer, no target network. Exploration comes from sampling (the policy is stochastic).
- Why `Softmax` at the end of the actor: outputs must be probabilities that sum to 1.
- Why we cannot update every step: `G` is not known until the episode ends.

**Common mistakes**: forgetting `.item()` in `env.step`; computing returns in forward order; using `torch.tensor(log_probs)` instead of `torch.stack(log_probs)` (it breaks the gradient).

**Expected output**: the average reward rises from about 20 toward 200 or more (noisy).

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.99
lr = 0.01
episodes = 500

# 2. STATES AND ACTIONS
env = gym.make("CartPole-v1")                           # state: 4 numbers, 2 actions
n_actions = 2
actor = nn.Sequential(
    nn.Linear(4, 128), nn.ReLU(),
    nn.Linear(128, n_actions), nn.Softmax(dim=-1)
)
optimizer = torch.optim.Adam(actor.parameters(), lr=lr)
totals = []

# 3. TRAINING LOOP
for episode in range(episodes):
    state, _ = env.reset()
    log_probs = []
    rewards = []
    done = False

    while not done:
        x = torch.tensor(state, dtype=torch.float32)

        # POLICY: actor
        dist = torch.distributions.Categorical(actor(x))
        action = dist.sample()
        log_probs.append(dist.log_prob(action))

        # ENVIRONMENT
        state, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        done = terminated or truncated

    # EQUATION: REINFORCE (after the episode)
    G = 0
    returns = []
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    loss = -(torch.stack(log_probs) * returns).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    totals.append(sum(rewards))
    if (episode + 1) % 50 == 0:
        print("Episode", episode + 1, "Avg reward (last 50):", sum(totals[-50:]) / 50)

# 4. OUTPUT
print("Avg reward, first 50 episodes:", sum(totals[:50]) / 50)
print("Avg reward, last 50 episodes: ", sum(totals[-50:]) / 50)

## Lab 10. Actor-Critic (One-Step, Update Every Step)

**Theory**
- REINFORCE has high variance and must wait for the episode to end. **Actor-Critic** fixes both by adding a second network:
  - **Actor**: the policy `pi(action | state)` (chooses actions).
  - **Critic**: the value function `V(state)` (judges how good a state is).
- The critic gives a **TD error** that is used as the **advantage**: `advantage = reward + gamma * V(next_state) - V(state)`.
  - Positive advantage: the action was better than expected, so make it more likely.
  - Negative advantage: worse than expected, so make it less likely.
- The critic learns by minimizing `advantage ** 2` (the TD(0) update from Lab 2, as a loss).
- It updates **every step** (bootstrapping), so it learns online without waiting for the episode to end.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | `Categorical(actor(x)).sample()` |
| ENVIRONMENT | `env.step(action.item())` |
| EQUATION (every step) | `advantage = r + gamma * V(next) * (1 - terminated) - V(state)`; `loss = -log_prob * advantage.detach() + advantage ** 2` |

**Keep / change vs Lab 9 (REINFORCE)**
- Keep: the actor network, `Categorical`, `log_prob`.
- Change: the update moves **inside** the step loop. No `G` list, no returns computation.
- Add: a `critic` network and the TD-error advantage.
- Why `.detach()` on the advantage in the actor term: the actor loss must not train the critic. The critic is trained only by `advantage ** 2`.
- Why `next_value.detach()`: the TD target is a label, so it must not receive gradient.

**Expected output**: the average reward rises (noisy). One-step updates are less stable than Lab 11.

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.99
lr = 0.003
episodes = 500

# 2. STATES AND ACTIONS
env = gym.make("CartPole-v1")                           # state: 4 numbers, 2 actions
n_actions = 2
actor = nn.Sequential(
    nn.Linear(4, 64), nn.ReLU(),
    nn.Linear(64, n_actions), nn.Softmax(dim=-1)
)
critic = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(
    list(actor.parameters()) + list(critic.parameters()), lr=lr
)
totals = []

# 3. TRAINING LOOP
for episode in range(episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        x = torch.tensor(state, dtype=torch.float32)

        # POLICY: actor
        dist = torch.distributions.Categorical(actor(x))
        action = dist.sample()

        # ENVIRONMENT
        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        next_x = torch.tensor(next_state, dtype=torch.float32)

        # EQUATION: Actor-Critic (every step)
        value = critic(x).squeeze()
        next_value = critic(next_x).squeeze().detach()
        target = reward + gamma * next_value * (1 - int(terminated))
        advantage = target - value

        loss = -dist.log_prob(action) * advantage.detach() + advantage ** 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state
        total_reward += reward

    totals.append(total_reward)
    if (episode + 1) % 50 == 0:
        print("Episode", episode + 1, "Avg reward (last 50):", sum(totals[-50:]) / 50)

# 4. OUTPUT
print("Avg reward, first 50 episodes:", sum(totals[:50]) / 50)
print("Avg reward, last 50 episodes: ", sum(totals[-50:]) / 50)

## Lab 11. Actor-Critic Methods: Advantage Actor-Critic (A2C)

**Theory**
- A2C keeps the actor and critic of Lab 10 but updates **once per episode (a batch)** using the **Monte Carlo return** `G` as the target, which is more stable than one-step updates.
- **Advantage** `A = G - V(state)`: how much better the actual return was than the critic expected. Using `V` as a **baseline** removes variance without adding bias.
- Loss has three parts:
  - **Actor loss**: `-log_prob * advantage.detach()`
  - **Critic loss**: `0.5 * advantage ** 2` (regression of `V` toward `G`)
  - **Entropy bonus**: `- 0.01 * entropy` keeps the policy from becoming deterministic too early (more exploration).
- "Methods" family: one-step Actor-Critic (Lab 10), A2C (this lab), A3C (A2C with many parallel workers), PPO (Lab 12, A2C with a safe clipped update).

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | `Categorical(actor(x)).sample()`, save `log_prob`, `value`, `entropy` |
| ENVIRONMENT | `env.step(action.item())`, save `reward` |
| EQUATION (after episode) | returns `G`, `advantage = G - value`, combined actor + critic + entropy loss |

**Keep / change**
- vs Lab 9 (REINFORCE): add a critic and use `G - V(state)` instead of `G`. This is "REINFORCE with a baseline".
- vs Lab 10 (Actor-Critic): update after the episode with real returns instead of every step with a bootstrapped target.
- Returns are normalized so the critic learns a small, stable scale.

**Expected output**: more stable improvement than Lab 10, average reward rising toward 200 or more.

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.99
lr = 0.003
episodes = 500

# 2. STATES AND ACTIONS
env = gym.make("CartPole-v1")                           # state: 4 numbers, 2 actions
n_actions = 2
actor = nn.Sequential(
    nn.Linear(4, 64), nn.ReLU(),
    nn.Linear(64, n_actions), nn.Softmax(dim=-1)
)
critic = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(
    list(actor.parameters()) + list(critic.parameters()), lr=lr
)
totals = []

# 3. TRAINING LOOP
for episode in range(episodes):
    state, _ = env.reset()
    log_probs, values, entropies, rewards = [], [], [], []
    done = False

    while not done:
        x = torch.tensor(state, dtype=torch.float32)

        # POLICY: actor (and critic's value, saved for the loss)
        dist = torch.distributions.Categorical(actor(x))
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        entropies.append(dist.entropy())
        values.append(critic(x).squeeze())

        # ENVIRONMENT
        state, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        done = terminated or truncated

    # EQUATION: A2C (after the episode)
    G = 0
    returns = []
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    log_probs = torch.stack(log_probs)
    values = torch.stack(values)
    entropy = torch.stack(entropies).mean()

    advantage = returns - values                        # A = G - V(state)
    actor_loss = -(log_probs * advantage.detach()).mean()
    critic_loss = (advantage ** 2).mean()
    loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    totals.append(sum(rewards))
    if (episode + 1) % 50 == 0:
        print("Episode", episode + 1, "Avg reward (last 50):", sum(totals[-50:]) / 50)

# 4. OUTPUT
print("Avg reward, first 50 episodes:", sum(totals[:50]) / 50)
print("Avg reward, last 50 episodes: ", sum(totals[-50:]) / 50)

## Lab 12. Proximal Policy Optimization (PPO)

**Theory**
- Problem with plain policy gradient: one big update can wreck the policy, and each episode of data is used only once.
- PPO lets us **reuse the same episode for several updates** while keeping every update **small and safe**.
- **Probability ratio**: `ratio = pi_new(a|s) / pi_old(a|s) = exp(new_log_prob - old_log_prob)`. Ratio 1 means no change.
- **Clipped objective**: `loss = -min(ratio * A, clip(ratio, 1-eps, 1+eps) * A)` with `eps = 0.2`. Once the ratio leaves `[0.8, 1.2]` in the helpful direction, the gradient is cut off, so the policy cannot move far from the old one.
- Everything else is A2C (Lab 11): actor, critic, advantage `A = G - V`, entropy bonus.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | actor samples action, **save the old log prob** (`.detach()`) |
| ENVIRONMENT | `env.step(action.item())` |
| EQUATION (after episode, repeated 4 times) | ratio, clipped surrogate, critic loss, entropy bonus |

**Exactly three changes from Lab 11 (marked `# PPO` in the code)**
1. `# PPO 1`: store `old_log_probs` (detached) while playing.
2. `# PPO 2`: repeat the update `for _ in range(4)` on the same episode, recomputing `new_log_probs` each time.
3. `# PPO 3`: use `ratio` and the **clipped** surrogate with `min`, instead of `log_prob * advantage`.

**Why `min`**: it makes the objective a pessimistic bound, so a lucky large ratio cannot give an over-optimistic update.

**Expected output**: average reward rises from about 20 toward 200 or more.

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.99
lr = 0.003
clip_eps = 0.2
epochs = 4                                              # PPO updates per episode
episodes = 300

# 2. STATES AND ACTIONS
env = gym.make("CartPole-v1")                           # state: 4 numbers, 2 actions
n_actions = 2
actor = nn.Sequential(
    nn.Linear(4, 64), nn.ReLU(),
    nn.Linear(64, n_actions), nn.Softmax(dim=-1)
)
critic = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(
    list(actor.parameters()) + list(critic.parameters()), lr=lr
)
totals = []

# 3. TRAINING LOOP
for episode in range(episodes):
    state, _ = env.reset()
    xs, actions, old_log_probs, rewards = [], [], [], []
    done = False

    while not done:
        x = torch.tensor(state, dtype=torch.float32)

        # POLICY: actor
        dist = torch.distributions.Categorical(actor(x))
        action = dist.sample()
        xs.append(x)
        actions.append(action)
        old_log_probs.append(dist.log_prob(action).detach())     # PPO 1: save old log prob

        # ENVIRONMENT
        state, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        done = terminated or truncated

    # EQUATION: PPO (after the episode)
    G = 0
    returns = []
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    xs = torch.stack(xs)
    actions = torch.stack(actions)
    old_log_probs = torch.stack(old_log_probs)

    for _ in range(epochs):                                      # PPO 2: several updates, same data
        dist = torch.distributions.Categorical(actor(xs))
        new_log_probs = dist.log_prob(actions)
        value = critic(xs).squeeze(-1)
        advantage = returns - value.detach()

        ratio = torch.exp(new_log_probs - old_log_probs)         # PPO 2: probability ratio
        surr1 = ratio * advantage
        surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantage   # PPO 3: clip
        actor_loss = -torch.min(surr1, surr2).mean()             # PPO 3: take the min
        critic_loss = ((returns - value) ** 2).mean()
        loss = actor_loss + 0.5 * critic_loss - 0.01 * dist.entropy().mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    totals.append(sum(rewards))
    if (episode + 1) % 50 == 0:
        print("Episode", episode + 1, "Avg reward (last 50):", sum(totals[-50:]) / 50)

# 4. OUTPUT
print("Avg reward, first 50 episodes:", sum(totals[:50]) / 50)
print("Avg reward, last 50 episodes: ", sum(totals[-50:]) / 50)

## Quick Revision: Same Template, Different Slots

| # | Algorithm | POLICY | EQUATION slot | Extra |
|---|-----------|--------|---------------|-------|
| 1 | Epsilon-greedy bandit | epsilon-greedy | `Q[a] += (r - Q[a]) / N[a]` | no state, no gamma |
| 2 | TD(0) state values | random | `V[s] += a*(r + g*V[s2] - V[s])` | `V` table, bootstraps |
| 3 | Policy Iteration | greedy on `V` | evaluate sweep, then `argmax` improve | needs the model, no sampling |
| 4 | Monte Carlo prediction | random | `V[s] += (G - V[s]) / N[s]` after the episode | no bootstrapping |
| 5 | Q-Learning | epsilon-greedy | `Q += a*(r + g*max Q[s2] - Q)` | off-policy |
| 6 | SARSA | epsilon-greedy, twice | `Q += a*(r + g*Q[s2,a2] - Q)` | on-policy |
| 7 | DQN | epsilon-greedy on `model` | `(Q(s,a) - (r + g*max Q_target(s2)))^2` | replay buffer + target net |
| 8 | TD-Gammon | max (X) / min (O) over afterstates | `w += a*delta*trace` | TD(lambda), self-play |
| 9 | REINFORCE | sample from actor | `loss = -(log_prob * G)` after the episode | returns, normalize |
| 10 | Actor-Critic | sample from actor | `adv = r + g*V(s2) - V(s)` every step | actor + critic |
| 11 | A2C | sample from actor | `adv = G - V(s)`, actor + 0.5 critic - entropy | per episode |
| 12 | PPO | sample from actor | `-min(ratio*adv, clip(ratio)*adv)` | old log probs, 4 epochs |

**Family tree (how each lab is a small edit of another)**
- `Lab 1 (bandit)` + state + `gamma*max Q[next]` -> `Lab 5 (Q-Learning)` -> change `max` to `Q[next, next_action]` -> `Lab 6 (SARSA)`.
- `Lab 5` with a network, replay and target network -> `Lab 7 (DQN)`.
- `Lab 2 (TD(0))` with a network and eligibility traces -> `Lab 8 (TD-Gammon)`.
- `Lab 4 (MC returns)` applied to a policy -> `Lab 9 (REINFORCE)` -> add a critic and TD error -> `Lab 10` -> use `G - V` and entropy per episode -> `Lab 11 (A2C)` -> add old log probs, ratio, clip, 4 epochs -> `Lab 12 (PPO)`.

**Categories**
- Table (NumPy): Labs 1-6. Network (PyTorch): Labs 7-12.
- Value-based: 2, 4, 5, 6, 7, 8. Policy-based: 9. Actor-critic (both): 10, 11, 12.
- On-policy: 6, 9, 10, 11, 12 (PPO is on-policy but reuses data for a few updates). Off-policy: 5, 7.
- Model-based (needs the model): 3. Model-free: all the others.
- Update timing: every step: 2, 5, 6, 7, 8, 10. After the episode: 4, 9, 11, 12.

## Viva Questions and One-Line Answers

- **Exploration vs exploitation?** Try new actions to learn vs use the best-known action to earn reward. Epsilon-greedy balances them.
- **Why discount (`gamma`)?** Near rewards matter more than far ones, and it keeps infinite sums finite.
- **TD vs Monte Carlo?** TD updates each step using a bootstrapped estimate (low variance, some bias). MC waits for the full return (no bias, high variance).
- **Q-Learning vs SARSA?** Q-Learning uses `max` of the next state (off-policy, learns the optimal policy). SARSA uses the actual next action (on-policy, learns the policy including its exploration).
- **Why does DQN need a replay buffer and a target network?** Replay breaks the correlation between samples. The target network keeps the learning target stable.
- **What does the critic do?** It estimates `V(state)` and produces the advantage that tells the actor whether an action was better than expected.
- **Why a baseline or advantage?** It lowers the variance of the policy gradient without adding bias.
- **What does PPO clipping do?** It limits how far the new policy can move from the old one in a single update, so training stays stable.
- **Policy iteration vs Q-Learning?** Policy iteration needs the full model and sweeps all states. Q-Learning learns from sampled experience with no model.
- **What is `lambda` in TD(lambda)?** It controls how far credit spreads back through time via eligibility traces: 0 = TD(0), 1 = Monte Carlo.

## Last-Minute Checklist and Common Bugs

- [ ] Q-table shape is `(n_states, n_actions)`; V-table is `(n_states,)`.
- [ ] Terminal state: the loop stops, and `V[terminal]` stays 0.
- [ ] SARSA: `next_action` chosen **before** the update; `action = next_action` at the end.
- [ ] REINFORCE / A2C / PPO: returns are computed **backward** with `reversed(rewards)`.
- [ ] Use `torch.stack(...)` for a list of tensors that need gradients.
- [ ] `.detach()` on targets, advantages in the actor loss, and old log probs.
- [ ] `env.reset()` returns `(state, info)`; `env.step()` returns 5 values; use `action.item()`.
- [ ] `optimizer.zero_grad()` -> `loss.backward()` -> `optimizer.step()` in this order.
- [ ] Print the output (table, path, average reward). An exam program with no output loses marks.
- [ ] If training is noisy, say so: RL results vary run to run. Fix a seed with `random.seed(0)`, `np.random.seed(0)`, `torch.manual_seed(0)` if the question asks for reproducibility.